# 数据压缩入门：教师演示

45分钟核心课。先预测，再按教师指令运行当前单元。不要提前运行全部单元。

需求：Python 3.10+、JupyterLab、Pillow；IPython由Jupyter提供。离线可运行，不需要widgets。整个课程目录一起复制。

教师控制：Q1隐藏精确比较，Q3才揭示；A1私有颜色卡只给发送者。Q5先收方案，再给A2。第40分钟进入Q10。失败20秒用PPTX静态页。课前清空全部输出。

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, HTML
here = Path.cwd().resolve()
candidates = [here, here / "1-2-encoding/1-2-4-data-compresson", *here.parents]
ROOT = next((p for p in candidates if (p / "demos/compression_lab.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请将整个课程文件夹复制到本机，并从该文件夹打开Notebook")
sys.path.insert(0, str(ROOT / "demos"))
from compression_lab import (COLORS, ALTERNATING, GRAY, rle_encode, rle_decode,
    named_runs, opening, predictive, approximate, image_evidence, save_jpeg_trial,
    video_changes, show_comparison_pair)
def show(name, width=384):
    display(Image(filename=str(ROOT / "assets/evidence" / name), width=width))
print("准备完成。按教师指令逐格运行，先写预测。")

## Q1 文件变小，是不是删掉了内容？ · D1

图中有哪些内容可能变化？写下你想检查的对象。

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

4096B→本机实际输出（随包为38B）。本阶段只观察外观，opening(True)留到Q3。不要把两PNG文件大小当颜色数据量。

先运行：原图与数据量。

In [ ]:
show("opening-original.png", 256)
opening(reveal=False)

收到预测后：只显示读回图，暂不检验精确相等。

In [ ]:
show("opening-restored.png", 256)

## Q2 连续重复的颜色，怎样少写一些？ · D2

发送者看私有颜色卡。记录规则、短消息、接收者还原结果；比较后再运行。

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

共享例红8绿5蓝3，payload 16B→6B。编号与次数各1B，双方预先共享规则。私有原串只给发送者，避免凭记忆还原。

活动后运行编码。

In [ ]:
encoded = rle_encode(COLORS)
print(named_runs(encoded))
print("编码的bytes：", list(encoded))

先预测还原，再运行。

In [ ]:
restored = rle_decode(encoded)
print("颜色顺序：", [int(v) for v in restored])

## Q3 少写以后，怎样证明一个也没丢？ · D1

提出一个精确检验方法。应比较哪两个对象？

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

4096B读回与输入逐byte一致。不能比较压缩bytes和原bytes，也不能仅看长度或外观。此时揭示无损定义。

核对方法后运行。

In [ ]:
print("RLE读回与输入一致：", rle_decode(rle_encode(COLORS)) == COLORS)
opening(reveal=True)

## Q4 同一规则，为什么有时反而变大？ · D2

两串都是16B；相同连续段规则，哪串更省？先数段，再运行。

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

16B→6B与16B→32B均能准确还原。无损不保证所有输入变小。不要把交替串改成红8绿8。

运行两组比较。

In [ ]:
for name, data in [("连续段", COLORS), ("红绿交替", ALTERNATING)]:
    result = rle_encode(data)
    print(name, "输入", len(data), "B；编码", len(result), "B；读回一致", rle_decode(result) == data)

## Q5 相邻值不相同，还能利用它们的关系吗？ · D3

原值120,121,122,121,120,120,121,122。先提规则，再填A2差值、读回末三项。

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

起点8bit＋7×2bit=22bit，装3B补2bit。−1=00，0=01，+1=10。长度、模式、码表未计入。差值没有少项；大差值拒绝，不能推广任意图像。

完成表后运行。

In [ ]:
values = list(GRAY)
predictive(values)

可选：学生提议出现大差值时，先预测能否使用本码表。

In [ ]:
try:
    print(predictive([120, 150]))
except ValueError as error:
    print(error)

## Q6 只留近似值，还能找回原数吗？ · D4

归最近的4的倍数，中点向上、上限252。写两个能恢复为120的原数。

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

120与121同归120。8×6=48bit=6B，还原120,120,124,120,120,120,120,124。此模型不是JPEG，不与初次采样量化混同。

完成A2近似表后运行。

In [ ]:
approximate(GRAY)

先将你想到的两个原数填入列表，再检验。

In [ ]:
candidate_values = []  # 填你想到的两个原数
if len(candidate_values) == 2:
    display(approximate(candidate_values))
else:
    print("先填两个原数，再运行检验。")

## Q7 看起来差不多，能否叫无损？ · D5

先比较中性A/B，再写：看不出变化足以证明数据相同吗？

预测／观察：________________

证据／判断：________________

**教师备课与预期结果**

同一源RGB，384×216。PNG4441B/像素一致；JPEGq80 10766B/28917像素变；q20 6752B/66885像素变。未压缩RGB248832B另列，不是PNG文件大小。quality范围是本库参数，不是通用质量百分比。只认主任务是否仍可获取。

只看图，先收判断。

In [ ]:
show_comparison_pair()

收判断后运行像素与文件大小检验。

In [ ]:
image_evidence()

同坐标放大比较。

In [ ]:
print("A：同坐标放大")
show("crop-source.png", 768)
print("B：同坐标放大")
show("crop-quality80.png", 768)

可选低质量对照；核心课赶时间时跳过。

In [ ]:
show("crop-quality20.png", 768)

可选：改quality并实测，只保存到demos/scratch。

In [ ]:
save_jpeg_trial(quality=80)

## Q8 同一种误差，哪些任务能接受？

A3：通知少了日期，程序少了等号（输入60），照片为清楚的浏览副本。逐项判断并写理由。

判断与理由：________________

教师结论：通知／源程序本体必须无损；照片题给浏览副本可有损，但换读小字或测量要重评。

## Q9 看扩展名，就能判定压缩方式吗？

依据PNG像素检验与TIFF配置卡，改正‘所有图片压缩都有损’。JPG、TIF、MP3、MPEG分别要补什么条件？

判断与理由：________________

教师结论：常见JPG照片通常有损；TIF看编码；MP3常见有损音频；MPEG是相关标准组，常见视频编码有损。PNG是图片无损反例。

## Q10 怎样在大小限制和信息要求之间选方案？

A3题目给定结果：限额100KiB；程序A80KiB/完整恢复、B50KiB/删改；照片A110KiB/像素一致、B70KiB/主题清楚。两项任务分别选择，并检查大小与信息要求。

判断与理由：________________

教师结论：程序选A，照片选B。先收40秒出口，课后核对：解压结果vs原输入；外观相似不证明原值一致。

## 出口检查

无损判断要比较________与________。

照片看不出变化仍可能有损，因为________。

## 第二课时可选：帧间变化

先预测：保存帧一之后，帧二要更新哪些格？一个2×2块右移一格。不是只保存新位置；旧位置还要清除。此模型不是完整MPEG码流。

In [ ]:
video_changes()

## D6音频拓展的实施边界

核心45分钟不做音频试听。本包未提供经统一增益、时延对齐的MP3听辨样本。第二课时若做D6，按course-design.qmd的8秒44.1kHz/16bit单声道PCM→128/32kbps规格，用同一源文件各编码一次，记录编码器、增益和时延；不得把音量差当作压缩差。来源与技术边界见课程设计C7。